In [ ]:
%pip install -r requirements.txt

In [ ]:
from pathlib import Path

print(f"Workspace local: {Path.cwd().resolve()}")


In [ ]:
from pathlib import Path
from time import perf_counter
from dataclasses import dataclass
import hashlib
import math
import re
import unicodedata

import chromadb
import numpy as np
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer


# ============================================================
# CONFIGURAÇÕES
# ============================================================

PASTA_DOCUMENTOS = Path("documentos")
PASTA_CHROMA = Path("chroma_db")


In [ ]:
NOME_COLECAO = "documentos_academicos_aula"

MODELO_EMBEDDING = (
    "sentence-transformers/"
    "paraphrase-multilingual-MiniLM-L12-v2"
)

TAMANHO_CHUNK = 800
OVERLAP_CHUNK = 150
TAMANHO_LOTE = 32
TOP_K = 3

# Modos disponíveis: "similaridade" ou "mmr".
MODO_BUSCA = "similaridade"

# No MMR, primeiro recuperamos um conjunto maior de candidatos.
FETCH_K_MMR = 12

# Próximo de 1: prioriza relevância.
# Menor: aumenta a diversidade.
LAMBDA_MMR = 0.6

# Na primeira execução desta versão, deixe True para recriar
# a coleção com os metadados de escopo e tipo de documento.
RECRIAR_COLECAO = True

# "baseline" mantém o comportamento da aula anterior.
# "avancado" executa o pipeline de engenharia de contexto.
MODO_PIPELINE = "avancado"

# Escopo opcional: "curso", "disciplina", "estagio" ou None.
# None é útil para verificar perguntas ambíguas.
ESCOPO_PADRAO = None

# Orçamento aproximado do contexto enviado ao LLM.
LIMITE_TOKENS_CONTEXTO = 700

# Limiar didático para eliminar candidatos fracos.
LIMIAR_SIMILARIDADE = 0.20

ESCOPOS_VALIDOS = {"curso", "disciplina", "estagio"}


# ============================================================
# UTILITÁRIOS DE NORMALIZAÇÃO
# ============================================================

def remover_acentos(texto: str) -> str:
    decomposed = unicodedata.normalize("NFD", texto)
    return "".join(
        char
        for char in decomposed
        if unicodedata.category(char) != "Mn"
    )


def normalizar_para_busca(texto: str) -> str:
    texto = remover_acentos(texto.lower())
    texto = re.sub(r"[^a-z0-9]+", " ", texto)
    return " ".join(texto.split())


def normalizar_texto(texto: str) -> str:
    texto = texto.replace("\u00a0", " ")
    texto = texto.replace("\r\n", "\n")
    texto = texto.replace("\r", "\n")
    texto = re.sub(r"[ \t]+", " ", texto)
    texto = re.sub(r"\n\s*\n+", "\n\n", texto)
    return texto.strip()


def normalizar_trecho_contexto(texto: str) -> str:
    """Normalização leve do contexto que será enviado ao LLM."""
    texto = texto.replace("\n", " ")
    texto = re.sub(r"\s+", " ", texto)
    return texto.strip()


def validar_escopo(escopo: str | None) -> None:
    if escopo is None:
        return

    if escopo not in ESCOPOS_VALIDOS:
        raise ValueError(
            f"Escopo inválido: {escopo}. "
            f"Use um de: {sorted(ESCOPOS_VALIDOS)}"
        )


# ============================================================
# INGESTÃO: LEITURA, ESCOPO E METADADOS
# ============================================================

def inferir_escopo(arquivo: str, texto: str = "") -> str:
    """
    Inferência didática de escopo.

    A decisão prioriza o nome do arquivo, que funciona como uma fonte
    simbólica/controlada. O texto é usado apenas como fallback.
    """
    nome = normalizar_para_busca(arquivo)

    # Primeiro: decisões controladas pelo nome/fonte do documento.
    if "estagio" in nome:
        return "estagio"

    if "ci1218" in nome or "disciplina" in nome or "ficha" in nome:
        return "disciplina"

    if "ppc" in nome or "projeto pedagogico" in nome:
        return "curso"

    # Fallback: conteúdo da página.
    conteudo = normalizar_para_busca(texto[:1200])

    if "ficha de disciplina" in conteudo or "ementa" in conteudo:
        return "disciplina"

    if "regulamento de estagio" in conteudo:
        return "estagio"

    if "projeto pedagogico" in conteudo or "ppc" in conteudo:
        return "curso"

    return "geral"


def inferir_tipo_documento(arquivo: str, texto: str = "") -> str:
    nome = normalizar_para_busca(arquivo)
    conteudo = normalizar_para_busca(texto[:1200])
    alvo = f"{nome} {conteudo}"

    if "regulamento" in alvo:
        return "regulamento"
    if "ficha" in alvo or "ementa" in alvo:
        return "ficha_disciplina"
    if "ppc" in alvo or "projeto pedagogico" in alvo:
        return "ppc"
    return "documento"


def carregar_pdfs(pasta: Path) -> list[dict]:
    if not pasta.exists():
        raise FileNotFoundError(
            f"A pasta não existe: {pasta.resolve()}"
        )

    arquivos = sorted(pasta.glob("*.pdf"))

    if not arquivos:
        raise FileNotFoundError(
            f"Nenhum PDF encontrado em: {pasta.resolve()}"
        )

    paginas = []

    for arquivo in arquivos:
        print(f"Lendo: {arquivo.name}")
        reader = PdfReader(str(arquivo))

        for numero_pagina, pagina in enumerate(reader.pages, start=1):
            texto = normalizar_texto(pagina.extract_text() or "")

            if not texto:
                print(
                    f"  Aviso: página {numero_pagina} "
                    "sem texto extraível."
                )
                continue

            paginas.append(
                {
                    "arquivo": arquivo.name,
                    "pagina": numero_pagina,
                    "texto": texto,
                    "escopo": inferir_escopo(arquivo.name, texto),
                    "tipo_documento": inferir_tipo_documento(
                        arquivo.name,
                        texto,
                    ),
                }
            )

    return paginas


# ============================================================
# CHUNKING
# ============================================================

def gerar_chunks(
    texto: str,
    tamanho: int = TAMANHO_CHUNK,
    overlap: int = OVERLAP_CHUNK,
) -> list[str]:
    if tamanho <= 0:
        raise ValueError("O tamanho do chunk deve ser maior que zero.")

    if overlap < 0:
        raise ValueError("O overlap não pode ser negativo.")

    if overlap >= tamanho:
        raise ValueError("O overlap deve ser menor que o tamanho.")

    chunks = []
    inicio = 0
    passo = tamanho - overlap

    while inicio < len(texto):
        fim = min(inicio + tamanho, len(texto))
        chunk = texto[inicio:fim].strip()

        if chunk:
            chunks.append(chunk)

        inicio += passo

    return chunks


def preparar_chunks(paginas: list[dict]) -> list[dict]:
    todos_chunks = []

    for pagina in paginas:
        chunks_da_pagina = gerar_chunks(
            pagina["texto"],
            tamanho=TAMANHO_CHUNK,
            overlap=OVERLAP_CHUNK,
        )

        nome_base = Path(pagina["arquivo"]).stem

        for numero_chunk, texto_chunk in enumerate(
            chunks_da_pagina,
            start=1,
        ):
            identificador = (
                f"{nome_base}"
                f"_p{pagina['pagina']:03d}"
                f"_c{numero_chunk:03d}"
            )

            todos_chunks.append(
                {
                    "id": identificador,
                    "arquivo": pagina["arquivo"],
                    "pagina": pagina["pagina"],
                    "numero_chunk": numero_chunk,
                    "texto": texto_chunk,
                    "escopo": pagina["escopo"],
                    "tipo_documento": pagina["tipo_documento"],
                }
            )

    return todos_chunks


# ============================================================
# BANCO VETORIAL
# ============================================================

def criar_colecao(recriar: bool):
    PASTA_CHROMA.mkdir(parents=True, exist_ok=True)

    client = chromadb.PersistentClient(path=str(PASTA_CHROMA))

    if recriar:
        try:
            client.delete_collection(NOME_COLECAO)
            print(f"Coleção anterior removida: {NOME_COLECAO}")
        except Exception:
            pass

    collection = client.get_or_create_collection(
        name=NOME_COLECAO,
        configuration={"hnsw": {"space": "cosine"}},
    )

    return collection


# ============================================================
# EMBEDDINGS E INDEXAÇÃO
# ============================================================

def indexar_chunks(
    collection,
    modelo: SentenceTransformer,
    chunks: list[dict],
) -> None:
    if not chunks:
        raise ValueError("Não existem chunks para indexar.")

    total = len(chunks)
    inicio_total = perf_counter()

    for inicio in range(0, total, TAMANHO_LOTE):
        fim = min(inicio + TAMANHO_LOTE, total)
        lote = chunks[inicio:fim]

        textos = [chunk["texto"] for chunk in lote]

        embeddings = modelo.encode(
            textos,
            batch_size=TAMANHO_LOTE,
            normalize_embeddings=True,
            show_progress_bar=False,
        )

        collection.upsert(
            ids=[chunk["id"] for chunk in lote],
            embeddings=embeddings.tolist(),
            documents=textos,
            metadatas=[
                {
                    "arquivo": chunk["arquivo"],
                    "pagina": chunk["pagina"],
                    "numero_chunk": chunk["numero_chunk"],
                    "tamanho_caracteres": len(chunk["texto"]),
                    "escopo": chunk["escopo"],
                    "tipo_documento": chunk["tipo_documento"],
                    "fonte_controlada": True,
                }
                for chunk in lote
            ],
        )

        print(f"Indexados: {fim}/{total}")

    tempo = perf_counter() - inicio_total

    print(f"Indexação concluída em {tempo:.2f} s.")
    print(f"Registros na coleção: {collection.count()}")


# ============================================================
# BUSCA VETORIAL BASELINE
# ============================================================

def buscar(
    collection,
    modelo: SentenceTransformer,
    pergunta: str,
    k: int = TOP_K,
    escopo: str | None = None,
) -> dict:
    pergunta = pergunta.strip()
    validar_escopo(escopo)

    if not pergunta:
        raise ValueError("A pergunta não pode estar vazia.")

    total_registros = collection.count()

    if total_registros == 0:
        raise RuntimeError("A coleção está vazia.")

    k_real = min(k, total_registros)

    embedding_pergunta = modelo.encode(
        pergunta,
        normalize_embeddings=True,
    ).tolist()

    inicio = perf_counter()

    parametros = {
        "query_embeddings": [embedding_pergunta],
        "n_results": k_real,
        "include": ["documents", "metadatas", "distances"],
    }

    # Filtro simbólico antes da busca vetorial.
    # Isso reduz o espaço de busca quando o escopo já é conhecido.
    if escopo is not None:
        parametros["where"] = {"escopo": escopo}

    resultados = collection.query(**parametros)

    resultados["latencia_ms"] = (perf_counter() - inicio) * 1000
    resultados["pergunta"] = pergunta
    resultados["metodo"] = "similaridade"
    resultados["escopo"] = escopo

    return resultados


# ============================================================
# MAXIMAL MARGINAL RELEVANCE
# ============================================================

def buscar_mmr(
    collection,
    modelo: SentenceTransformer,
    pergunta: str,
    k: int = TOP_K,
    fetch_k: int = FETCH_K_MMR,
    lambda_mult: float = LAMBDA_MMR,
    escopo: str | None = None,
) -> dict:
    """
    Recupera fetch_k candidatos por similaridade e seleciona k
    resultados equilibrando relevância e diversidade.
    """
    pergunta = pergunta.strip()
    validar_escopo(escopo)

    if not pergunta:
        raise ValueError("A pergunta não pode estar vazia.")
    if k <= 0:
        raise ValueError("k deve ser maior que zero.")
    if fetch_k < k:
        raise ValueError("fetch_k deve ser maior ou igual a k.")
    if not 0 <= lambda_mult <= 1:
        raise ValueError("lambda_mult deve estar entre 0 e 1.")

    total_registros = collection.count()

    if total_registros == 0:
        raise RuntimeError("A coleção está vazia.")

    k_real = min(k, total_registros)
    fetch_k_real = min(max(fetch_k, k_real), total_registros)

    embedding_pergunta = modelo.encode(
        pergunta,
        normalize_embeddings=True,
    )

    inicio = perf_counter()

    parametros = {
        "query_embeddings": [embedding_pergunta.tolist()],
        "n_results": fetch_k_real,
        "include": [
            "documents",
            "metadatas",
            "distances",
            "embeddings",
        ],
    }

    if escopo is not None:
        parametros["where"] = {"escopo": escopo}

    candidatos = collection.query(**parametros)

    ids = candidatos["ids"][0]
    documentos = candidatos["documents"][0]
    metadados = candidatos["metadatas"][0]

    if not ids:
        return {
            "ids": [[]],
            "documents": [[]],
            "metadatas": [[]],
            "distances": [[]],
            "latencia_ms": (perf_counter() - inicio) * 1000,
            "pergunta": pergunta,
            "metodo": "MMR",
            "lambda_mmr": lambda_mult,
            "fetch_k": fetch_k_real,
            "escopo": escopo,
        }

    embeddings = np.asarray(
        candidatos["embeddings"][0],
        dtype=np.float32,
    )

    relevancias = embeddings @ embedding_pergunta
    selecionados = [int(np.argmax(relevancias))]

    while len(selecionados) < k_real and len(selecionados) < len(ids):
        melhor_indice = None
        melhor_score = float("-inf")

        for indice in range(len(ids)):
            if indice in selecionados:
                continue

            redundancias = embeddings[selecionados] @ embeddings[indice]
            maior_redundancia = float(np.max(redundancias))

            score_mmr = (
                lambda_mult * float(relevancias[indice])
                - (1 - lambda_mult) * maior_redundancia
            )

            if score_mmr > melhor_score:
                melhor_score = score_mmr
                melhor_indice = indice

        if melhor_indice is None:
            break

        selecionados.append(melhor_indice)

    latencia_ms = (perf_counter() - inicio) * 1000

    similaridades_selecionadas = [
        float(relevancias[indice])
        for indice in selecionados
    ]

    return {
        "ids": [[ids[indice] for indice in selecionados]],
        "documents": [[documentos[indice] for indice in selecionados]],
        "metadatas": [[metadados[indice] for indice in selecionados]],
        "distances": [[
            1 - similaridade
            for similaridade in similaridades_selecionadas
        ]],
        "latencia_ms": latencia_ms,
        "pergunta": pergunta,
        "metodo": "MMR",
        "lambda_mmr": lambda_mult,
        "fetch_k": fetch_k_real,
        "escopo": escopo,
    }


# ============================================================
# EXIBIÇÃO DO BASELINE
# ============================================================

def imprimir_resultados(resultados: dict) -> None:
    documentos = resultados["documents"][0]
    metadados = resultados["metadatas"][0]
    distancias = resultados["distances"][0]
    ids = resultados["ids"][0]

    print("\n" + "#" * 80)
    print(f"Pergunta: {resultados['pergunta']}")
    print(f"Método: {resultados.get('metodo', 'similaridade')}")
    print(f"Escopo: {resultados.get('escopo')}")
    print(f"Latência da busca: {resultados['latencia_ms']:.2f} ms")
    print("#" * 80)

    for rank, (identificador, texto, metadata, distancia) in enumerate(
        zip(ids, documentos, metadados, distancias),
        start=1,
    ):
        similaridade = 1 - distancia

        print("\n" + "=" * 80)
        print(f"Rank: {rank}")
        print(f"ID: {identificador}")
        print(f"Arquivo: {metadata.get('arquivo')}")
        print(f"Página: {metadata.get('pagina')}")
        print(f"Chunk: {metadata.get('numero_chunk')}")
        print(f"Escopo: {metadata.get('escopo', 'nao_informado')}")
        print(f"Tipo: {metadata.get('tipo_documento', 'nao_informado')}")
        print(f"Distância de cosseno: {distancia:.4f}")
        print(f"Similaridade aproximada: {similaridade:.4f}")
        print("-" * 80)
        print(texto[:1200])


# ============================================================
# AULA 2: ENGENHARIA DE CONTEXTO
# ============================================================

@dataclass
class ResultadoChunk:
    id: str
    texto: str
    metadata: dict
    distancia: float
    similaridade: float
    consulta_origem: str


def estimar_tokens(texto: str) -> int:
    """
    Estimativa didática: 1 token ~= 4 caracteres.
    Para produção, use o tokenizer do modelo escolhido.
    """
    return math.ceil(len(texto) / 4)


def estimar_tokens_resultados(resultados: dict) -> int:
    documentos = resultados["documents"][0]
    return sum(estimar_tokens(documento) for documento in documentos)


# ------------------------------------------------------------
# Pré-recuperação: ambiguidade e transformação de consultas
# ------------------------------------------------------------

def detectar_ambiguidade(pergunta: str) -> bool:
    p = normalizar_para_busca(pergunta)
    termos_ambiguos = [
        "frequencia minima",
        "carga horaria",
        "horas obrigatorias",
    ]
    return any(termo in p for termo in termos_ambiguos)


def reescrever_consulta(
    pergunta: str,
    escopo: str | None = None,
) -> str:
    validar_escopo(escopo)
    p = normalizar_para_busca(pergunta)

    if "frequencia" in p:
        if escopo == "disciplina":
            return (
                "frequencia minima para aprovacao "
                "em disciplina regular"
            )

        if escopo == "estagio":
            return (
                "regra de frequencia ou assiduidade "
                "no estagio obrigatorio"
            )

        if escopo == "curso":
            return (
                "regra geral de frequencia minima "
                "para aprovacao prevista no PPC do curso"
            )

        # Sem escopo não assumimos interpretação.
        return pergunta

    if "sql" in p:
        return (
            "conteudo programatico da disciplina "
            "de banco de dados SQL"
        )

    if "estagio" in p:
        return (
            "carga horaria do estagio obrigatorio "
            "supervisionado"
        )

    return pergunta


def expandir_consulta(
    pergunta: str,
    escopo: str | None = None,
) -> list[str]:
    validar_escopo(escopo)
    p = normalizar_para_busca(pergunta)

    if "frequencia" in p:
        if escopo == "disciplina":
            return [
                pergunta,
                "presenca minima para aprovacao em disciplina",
                "percentual minimo de comparecimento disciplina",
                "limite de faltas disciplina regular",
            ]

        if escopo == "estagio":
            return [
                pergunta,
                "frequencia no estagio obrigatorio",
                "assiduidade no estagio supervisionado",
                "controle de presenca no estagio obrigatorio",
            ]

        if escopo == "curso":
            return [
                pergunta,
                "regra geral de frequencia prevista no PPC",
                "criterios de aprovacao por frequencia no curso",
                "normas academicas de frequencia do curso",
            ]

        # Pergunta ainda ambígua: não expandir agressivamente.
        return [pergunta]

    if "estagio" in p:
        return [
            pergunta,
            "carga horaria estagio obrigatorio",
            "horas estagio supervisionado",
        ]

    return [pergunta]


def step_back(
    pergunta: str,
    escopo: str | None = None,
) -> str:
    validar_escopo(escopo)
    p = normalizar_para_busca(pergunta)

    if "frequencia" in p:
        if escopo == "disciplina":
            return (
                "regras academicas de aprovacao "
                "e frequencia em disciplinas"
            )

        if escopo == "estagio":
            return (
                "regras academicas de acompanhamento "
                "e assiduidade no estagio"
            )

        if escopo == "curso":
            return (
                "normas academicas gerais de aprovacao "
                "e frequencia previstas no PPC"
            )

        return "regras academicas de frequencia"

    if "estagio" in p:
        return (
            "regras dos componentes curriculares "
            "obrigatorios do curso"
        )

    return pergunta


def hyde_controlado(
    pergunta: str,
    escopo: str | None = None,
) -> str:
    """
    Simulação didática de HyDE sem chamar um LLM.

    O texto hipotético serve somente para produzir uma consulta de
    recuperação. Ele NÃO é evidência.
    """
    validar_escopo(escopo)
    p = normalizar_para_busca(pergunta)

    if "estagio" in p and "frequencia" not in p:
        return (
            "O regulamento de estágio informa "
            "a carga horária do estágio obrigatório."
        )

    if "frequencia" in p:
        if escopo == "disciplina":
            return (
                "A ficha ou norma acadêmica informa "
                "a frequência mínima necessária para "
                "aprovação em uma disciplina."
            )

        if escopo == "estagio":
            return (
                "O regulamento de estágio estabelece "
                "regras de frequência e assiduidade "
                "para o estágio obrigatório."
            )

        if escopo == "curso":
            return (
                "O projeto pedagógico do curso apresenta "
                "as normas gerais de frequência e os "
                "critérios acadêmicos de aprovação."
            )

        return pergunta

    return pergunta


def gerar_consultas(
    pergunta: str,
    escopo: str | None = None,
) -> list[str]:
    validar_escopo(escopo)

    consultas = [
        pergunta,
        reescrever_consulta(pergunta, escopo=escopo),
    ]

    consultas.extend(
        expandir_consulta(pergunta, escopo=escopo)
    )

    consultas.append(
        step_back(pergunta, escopo=escopo)
    )

    consultas.append(
        hyde_controlado(pergunta, escopo=escopo)
    )

    # Remove duplicatas preservando a ordem.
    return list(dict.fromkeys(consultas))


# ------------------------------------------------------------
# Recuperação e fusão
# ------------------------------------------------------------

def converter_resultados(
    resultados: dict,
    consulta: str,
) -> list[ResultadoChunk]:
    itens = []

    for identificador, documento, metadata, distancia in zip(
        resultados["ids"][0],
        resultados["documents"][0],
        resultados["metadatas"][0],
        resultados["distances"][0],
    ):
        itens.append(
            ResultadoChunk(
                id=identificador,
                texto=documento,
                metadata=metadata,
                distancia=float(distancia),
                similaridade=1 - float(distancia),
                consulta_origem=consulta,
            )
        )

    return itens


def buscar_lista(
    collection,
    modelo: SentenceTransformer,
    pergunta: str,
    k: int = TOP_K,
    escopo: str | None = None,
) -> list[ResultadoChunk]:
    resultados = buscar(
        collection,
        modelo,
        pergunta,
        k=k,
        escopo=escopo,
    )
    return converter_resultados(resultados, consulta=pergunta)


def fusao_rrf(
    listas: list[list[ResultadoChunk]],
    k_rrf: int = 60,
) -> list[ResultadoChunk]:
    scores = {}
    objetos = {}

    for lista in listas:
        for posicao, item in enumerate(lista, start=1):
            scores[item.id] = (
                scores.get(item.id, 0.0)
                + 1.0 / (k_rrf + posicao)
            )
            objetos[item.id] = item

    ordenados = sorted(
        scores.items(),
        key=lambda par: par[1],
        reverse=True,
    )

    return [
        objetos[identificador]
        for identificador, _ in ordenados
    ]


def multi_busca(
    collection,
    modelo: SentenceTransformer,
    pergunta: str,
    escopo: str | None = None,
    k: int = 4,
) -> list[ResultadoChunk]:
    validar_escopo(escopo)
    listas = []

    consultas = gerar_consultas(
        pergunta,
        escopo=escopo,
    )

    for consulta in consultas:
        listas.append(
            buscar_lista(
                collection,
                modelo,
                consulta,
                k=k,
                escopo=escopo,
            )
        )

    return fusao_rrf(listas)


# ------------------------------------------------------------
# Pós-recuperação: filtros, deduplicação e reranking
# ------------------------------------------------------------

def filtrar_por_escopo(
    candidatos: list[ResultadoChunk],
    escopo: str | None,
) -> list[ResultadoChunk]:
    """
    Filtro defensivo pós-recuperação.

    No pipeline avançado, o escopo também é usado como filtro simbólico
    durante a própria busca no ChromaDB. Mantemos esta etapa para tornar
    explícita a validação do contexto antes de enviá-lo ao LLM.
    """
    validar_escopo(escopo)

    if escopo is None:
        return candidatos

    return [
        candidato
        for candidato in candidatos
        if candidato.metadata.get("escopo") == escopo
    ]


def filtrar_por_score(
    candidatos: list[ResultadoChunk],
    minimo: float = LIMIAR_SIMILARIDADE,
) -> list[ResultadoChunk]:
    return [
        candidato
        for candidato in candidatos
        if candidato.similaridade >= minimo
    ]


def hash_texto(texto: str) -> str:
    normalizado = normalizar_trecho_contexto(
        normalizar_para_busca(texto)
    )
    return hashlib.md5(
        normalizado.encode("utf-8")
    ).hexdigest()[:12]


def deduplicar_textual(
    candidatos: list[ResultadoChunk],
) -> list[ResultadoChunk]:
    vistos = set()
    saida = []

    for candidato in candidatos:
        h = hash_texto(candidato.texto)

        if h not in vistos:
            vistos.add(h)
            saida.append(candidato)

    return saida


def termos_conteudo(texto: str) -> set[str]:
    stopwords = {
        "qual", "quais", "como", "para", "possui", "sobre",
        "uma", "um", "de", "do", "da", "das", "dos", "e",
        "o", "a", "as", "os", "em", "no", "na", "nos", "nas",
        "que", "se", "ao", "aos", "ou", "por", "com",
    }

    termos = re.findall(
        r"\w+",
        normalizar_para_busca(texto),
    )

    return {
        termo
        for termo in termos
        if termo not in stopwords and len(termo) > 2
    }


def deduplicar_semantico_lexical(
    candidatos: list[ResultadoChunk],
    limiar_jaccard: float = 0.82,
) -> list[ResultadoChunk]:
    """
    Deduplicação aproximada para a demonstração.

    Usa sobreposição lexical. Em produção, pode ser substituída por
    similaridade entre embeddings dos próprios chunks.
    """
    saida = []
    assinaturas = []

    for candidato in candidatos:
        termos = termos_conteudo(candidato.texto)
        duplicado = False

        for assinatura in assinaturas:
            inter = len(termos & assinatura)
            union = len(termos | assinatura) or 1
            jaccard = inter / union

            if jaccard >= limiar_jaccard:
                duplicado = True
                break

        if not duplicado:
            saida.append(candidato)
            assinaturas.append(termos)

    return saida


def deduplicar(
    candidatos: list[ResultadoChunk],
) -> list[ResultadoChunk]:
    candidatos = deduplicar_textual(candidatos)
    candidatos = deduplicar_semantico_lexical(candidatos)
    return candidatos


def palavras_relevantes(pergunta: str) -> set[str]:
    return termos_conteudo(pergunta)


def rerank_lexical(
    pergunta: str,
    candidatos: list[ResultadoChunk],
) -> list[ResultadoChunk]:
    """
    Reranker didático sem modelo adicional.

    Combina similaridade vetorial, cobertura lexical e pequenos bônus
    por presença de fonte/metadado controlado.
    """
    termos = palavras_relevantes(pergunta)

    def score(candidato: ResultadoChunk) -> float:
        texto = normalizar_para_busca(candidato.texto)
        cobertura = sum(
            1
            for termo in termos
            if termo in texto
        )

        bonus_fonte = (
            0.05
            if candidato.metadata.get("arquivo")
            else 0.0
        )

        bonus_controlado = (
            0.05
            if candidato.metadata.get("fonte_controlada")
            else 0.0
        )

        return (
            candidato.similaridade
            + 0.12 * cobertura
            + bonus_fonte
            + bonus_controlado
        )

    return sorted(
        candidatos,
        key=score,
        reverse=True,
    )


# ------------------------------------------------------------
# Compressão de contexto
# ------------------------------------------------------------

def sentencas_relevantes(
    pergunta: str,
    texto: str,
    limite: int = 3,
) -> list[str]:
    termos = palavras_relevantes(pergunta)
    texto = normalizar_trecho_contexto(texto)

    sentencas = re.split(
        r"(?<=[.!?])\s+",
        texto.strip(),
    )

    selecionadas = [
        sentenca
        for sentenca in sentencas
        if any(
            termo in normalizar_para_busca(sentenca)
            for termo in termos
        )
    ]

    return selecionadas[:limite] or sentencas[:1]


def montar_prompt_compressao_abstrativa(
    pergunta: str,
    trecho: str,
) -> str:
    """
    Prompt de exemplo para uma compressão abstrativa com LLM.

    A demonstração executa compressão extrativa para não depender de API.
    """
    return f"""
Resuma o TRECHO apenas para responder à PERGUNTA.
Preserve números, condições, exceções e fonte.
Não acrescente informação externa.
Se o trecho não ajudar, responda: IRRELEVANTE.

PERGUNTA:
{pergunta}

TRECHO:
{trecho}
""".strip()


def comprimir_contexto(
    pergunta: str,
    candidatos: list[ResultadoChunk],
    limite_tokens: int = LIMITE_TOKENS_CONTEXTO,
) -> tuple[list[str], int]:
    contexto = []
    usados = 0

    for candidato in candidatos:
        trecho = " ".join(
            sentencas_relevantes(
                pergunta,
                candidato.texto,
            )
        )

        trecho = normalizar_trecho_contexto(trecho)

        fonte = (
            f"{candidato.metadata.get('arquivo')} "
            f"p.{candidato.metadata.get('pagina')} "
            f"escopo={candidato.metadata.get('escopo')}"
        )

        item = f"Fonte: {fonte}. Trecho: {trecho}"
        custo = estimar_tokens(item)

        if usados + custo <= limite_tokens:
            contexto.append(item)
            usados += custo

    return contexto, usados


# ------------------------------------------------------------
# Contexto suficiente e decisão de resposta
# ------------------------------------------------------------

def contexto_suficiente(
    pergunta: str,
    contexto: list[str],
) -> tuple[bool, str]:
    if not contexto:
        return False, "nenhum contexto foi selecionado"

    texto_contexto = " ".join(contexto)
    texto_normalizado = normalizar_para_busca(texto_contexto)
    termos = palavras_relevantes(pergunta)

    cobertura = sum(
        1
        for termo in termos
        if termo in texto_normalizado
    )

    if cobertura == 0:
        return (
            False,
            "o contexto não cobre os termos centrais da pergunta",
        )

    p = normalizar_para_busca(pergunta)

    # Perguntas quantitativas precisam de um valor explícito.
    if any(
        termo in p
        for termo in [
            "quantas",
            "quanto",
            "carga horaria",
            "horas",
            "frequencia",
        ]
    ):
        possui_numero = bool(
            re.search(r"\b\d+(?:[.,]\d+)?\b", texto_contexto)
        )
        possui_percentual = "%" in texto_contexto

        if not (possui_numero or possui_percentual):
            return (
                False,
                "a pergunta exige um valor, mas o contexto não contém número ou percentual",
            )

    return True, "há evidência mínima para responder"


def montar_prompt(
    pergunta: str,
    contexto: list[str],
) -> str:
    evidencias = "\n".join(
        f"[{indice}] {conteudo}"
        for indice, conteudo in enumerate(
            contexto,
            start=1,
        )
    )

    return f"""
Responda somente com base no CONTEXTO.
Se o contexto não for suficiente, diga que não há evidência suficiente.
Não complete lacunas com conhecimento externo.
Cite a fonte usada.

PERGUNTA:
{pergunta}

CONTEXTO:
{evidencias}
""".strip()


def sugerir_fonte_estruturada(pergunta: str) -> str:
    p = normalizar_para_busca(pergunta)

    if "frequencia" in p or "carga horaria" in p or "estagio" in p:
        return "metadados/regras acadêmicas estruturadas"

    if "disciplina" in p or "sql" in p:
        return "consulta SQL no catálogo de disciplinas"

    return "documentos recuperados + metadados controlados"


# ------------------------------------------------------------
# Exibição das transformações
# ------------------------------------------------------------

def imprimir_consultas_geradas(
    pergunta: str,
    escopo: str | None,
) -> None:
    print("\nConsultas que serão executadas:")

    for indice, consulta in enumerate(
        gerar_consultas(pergunta, escopo),
        start=1,
    ):
        print(f"  {indice}. {consulta}")


# ------------------------------------------------------------
# Pipeline final com medição de custo por etapa
# ------------------------------------------------------------

def responder_com_contexto(
    collection,
    modelo: SentenceTransformer,
    pergunta: str,
    escopo: str | None = None,
) -> str:
    validar_escopo(escopo)
    pergunta = pergunta.strip()

    if not pergunta:
        raise ValueError("A pergunta não pode estar vazia.")

    # --------------------------------------------------------
    # 1. Decisão ANTES da recuperação
    # --------------------------------------------------------
    if detectar_ambiguidade(pergunta) and escopo is None:
        return (
            "Acao: esclarecer\n"
            "Motivo: pergunta ambigua sem escopo definido\n\n"
            "A pergunta pode se referir a diferentes regras.\n"
            "Voce se refere a:\n"
            "1. disciplina;\n"
            "2. estagio;\n"
            "3. regra geral prevista no PPC do curso?"
        )

    tempo_total = perf_counter()
    tempos = {}

    # --------------------------------------------------------
    # 2. Transformação + múltiplas buscas + RRF
    # --------------------------------------------------------
    t = perf_counter()

    candidatos = multi_busca(
        collection,
        modelo,
        pergunta,
        escopo=escopo,
        k=4,
    )

    tempos["multi_busca_rrf_ms"] = (
        perf_counter() - t
    ) * 1000

    total_pos_fusao = len(candidatos)

    # --------------------------------------------------------
    # 3. Filtros
    # --------------------------------------------------------
    t = perf_counter()

    candidatos = filtrar_por_escopo(
        candidatos,
        escopo,
    )

    candidatos = filtrar_por_score(
        candidatos,
        minimo=LIMIAR_SIMILARIDADE,
    )

    tempos["filtros_ms"] = (
        perf_counter() - t
    ) * 1000

    total_pos_filtros = len(candidatos)

    # --------------------------------------------------------
    # 4. Deduplicação
    # --------------------------------------------------------
    t = perf_counter()

    candidatos = deduplicar(candidatos)

    tempos["deduplicacao_ms"] = (
        perf_counter() - t
    ) * 1000

    total_pos_dedup = len(candidatos)

    # --------------------------------------------------------
    # 5. Reranking
    # --------------------------------------------------------
    t = perf_counter()

    candidatos = rerank_lexical(
        pergunta,
        candidatos,
    )

    tempos["reranking_ms"] = (
        perf_counter() - t
    ) * 1000

    # --------------------------------------------------------
    # 6. Compressão
    # --------------------------------------------------------
    t = perf_counter()

    contexto, tokens = comprimir_contexto(
        pergunta,
        candidatos[:5],
        limite_tokens=LIMITE_TOKENS_CONTEXTO,
    )

    tempos["compressao_ms"] = (
        perf_counter() - t
    ) * 1000

    # --------------------------------------------------------
    # 7. Contexto suficiente?
    # --------------------------------------------------------
    suficiente, motivo = contexto_suficiente(
        pergunta,
        contexto,
    )

    latencia_ms = (
        perf_counter() - tempo_total
    ) * 1000

    linhas_tempo = "\n".join(
        f"- {nome}: {valor:.2f} ms"
        for nome, valor in tempos.items()
    )

    cabecalho = (
        f"Escopo: {escopo}\n"
        f"Candidatos apos fusao: {total_pos_fusao}\n"
        f"Candidatos apos filtros: {total_pos_filtros}\n"
        f"Candidatos apos deduplicacao: {total_pos_dedup}\n"
        f"Tokens aproximados do contexto: {tokens}\n"
        f"Latencia total aproximada: {latencia_ms:.2f} ms\n"
        f"Fonte estruturada sugerida: "
        f"{sugerir_fonte_estruturada(pergunta)}\n"
        f"\nTempos por etapa:\n{linhas_tempo}\n"
    )

    if not suficiente:
        return (
            cabecalho
            + "\nAcao: abster\n"
            + f"Motivo: {motivo}\n"
            + "Nao ha evidencia suficiente no contexto recuperado."
        )

    prompt = montar_prompt(
        pergunta,
        contexto,
    )

    return (
        cabecalho
        + "\nAcao: responder\n"
        + f"Motivo: {motivo}\n\n"
        + prompt
    )


# ============================================================
# PROGRAMA PRINCIPAL
# ============================================================

def main() -> None:
    try:
        print("1. Carregando PDFs...")
        paginas = carregar_pdfs(PASTA_DOCUMENTOS)
        print(f"Páginas com texto: {len(paginas)}")

        print("\n2. Gerando chunks...")
        chunks = preparar_chunks(paginas)
        print(f"Total de chunks: {len(chunks)}")

        print("\n3. Carregando o modelo de embeddings...")
        modelo = SentenceTransformer(MODELO_EMBEDDING)

        print(f"Modelo: {MODELO_EMBEDDING}")
        print(
            "Dimensão dos embeddings: "
            f"{modelo.get_sentence_embedding_dimension()}"
        )

        print("\n4. Preparando a coleção...")
        collection = criar_colecao(
            recriar=RECRIAR_COLECAO
        )

        if RECRIAR_COLECAO or collection.count() == 0:
            print("\n5. Indexando os chunks...")
            indexar_chunks(
                collection,
                modelo,
                chunks,
            )
        else:
            print(
                "\nColeção persistente reutilizada. "
                f"Registros: {collection.count()}"
            )

        print("\n6. Busca interativa")
        print("Comandos disponíveis:")
        print("  modo baseline")
        print("  modo avancado")
        print("  escopo disciplina")
        print("  escopo estagio")
        print("  escopo curso")
        print("  escopo nenhum")
        print("  mostrar consultas")
        print("  sair")

        modo_pipeline = MODO_PIPELINE
        escopo_atual = ESCOPO_PADRAO
        mostrar_consultas = True

        while True:
            pergunta = input(
                "\nDigite sua opção ou pergunta ou 'sair': "
            ).strip()

            if pergunta.lower() == "sair":
                print("Programa encerrado.")
                break

            comando = normalizar_para_busca(pergunta)

            if comando == "modo baseline":
                modo_pipeline = "baseline"
                print("Modo alterado para baseline.")
                continue

            if comando == "modo avancado":
                modo_pipeline = "avancado"
                print("Modo alterado para avançado.")
                continue

            if comando == "mostrar consultas":
                mostrar_consultas = not mostrar_consultas
                print(
                    "Exibição das consultas geradas: "
                    f"{mostrar_consultas}"
                )
                continue

            if comando.startswith("escopo "):
                valor = comando.replace(
                    "escopo ",
                    "",
                    1,
                ).strip()

                novo_escopo = (
                    None
                    if valor == "nenhum"
                    else valor
                )

                validar_escopo(novo_escopo)
                escopo_atual = novo_escopo
                print(f"Escopo atual: {escopo_atual}")
                continue

            if not pergunta:
                print("Digite uma pergunta válida.")
                continue

            # ------------------------------------------------
            # PIPELINE AVANÇADO
            # ------------------------------------------------
            if modo_pipeline == "avancado":
                if (
                    mostrar_consultas
                    and not (
                        detectar_ambiguidade(pergunta)
                        and escopo_atual is None
                    )
                ):
                    imprimir_consultas_geradas(
                        pergunta,
                        escopo_atual,
                    )

                print(
                    responder_com_contexto(
                        collection,
                        modelo,
                        pergunta,
                        escopo=escopo_atual,
                    )
                )
                continue

            # ------------------------------------------------
            # BASELINE DA AULA ANTERIOR
            # ------------------------------------------------
            if MODO_BUSCA == "mmr":
                resultados = buscar_mmr(
                    collection,
                    modelo,
                    pergunta,
                    k=TOP_K,
                    fetch_k=FETCH_K_MMR,
                    lambda_mult=LAMBDA_MMR,
                    escopo=None,
                )
            else:
                resultados = buscar(
                    collection,
                    modelo,
                    pergunta,
                    k=TOP_K,
                    escopo=None,
                )

            imprimir_resultados(resultados)

            print(
                "Tokens aproximados dos chunks recuperados: "
                f"{estimar_tokens_resultados(resultados)}"
            )

    except KeyboardInterrupt:
        print("\nPrograma interrompido.")

    except (
        FileNotFoundError,
        ValueError,
        RuntimeError,
    ) as erro:
        print(f"\nErro: {erro}")

    except Exception as erro:
        print(
            "\nErro inesperado: "
            f"{type(erro).__name__}: {erro}"
        )


if __name__ == "__main__":
    main()
